In [1]:
import os
os.chdir('/home/asudupe/Latxa-Omni/')

In [2]:
import torch
import torchaudio
from omni_speech.constants import SPEECH_TOKEN_INDEX, DEFAULT_SPEECH_TOKEN
from omni_speech.conversation import conv_templates, SeparatorStyle
from omni_speech.model.builder import load_pretrained_model
from omni_speech.datasets.preprocess import tokenizer_speech_token
from torch.utils.data import Dataset, DataLoader
import whisper
from datasets import load_dataset, load_from_disk
import numpy as np
from IPython.display import Audio
from scipy.io.wavfile import write
from torchaudio.transforms import Resample
from speechbrain.inference.vocoders import UnitHIFIGAN
from transformers import Wav2Vec2Processor, AutoTokenizer

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def ctc_postprocess(tokens, blank):
    _toks = tokens.squeeze(0).tolist()
    deduplicated_toks = [v for i, v in enumerate(_toks) if i == 0 or v != _toks[i - 1]]
    hyp = [v for v in deduplicated_toks if v != blank] #官方493 222
    hyp = " ".join(list(map(str, hyp))) #1918 547
    return hyp

In [16]:
dataset = load_from_disk('/scratch/asudupe/datasets/VoiceAssistant-400K_eu/')

In [ ]:
speech, sr = torchaudio.load(os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['train'][10]['question_audio']))
Audio(data=np.array(speech), rate=sr)

In [ ]:
dataset['train'][10]['answer']

In [ ]:
write(filename='example.wav', data=np.array(dataset['train'][4]['question_audio'], dtype=np.float32), rate=22050)

In [ ]:
speech_file = "omni_speech/serve/examples/helpful_base_1.wav"
speech = whisper.load_audio(speech_file)

Audio(data=np.array(speech), rate=16000)


In [4]:
# model_path = 'saves/13834/checkpoint-24000'
# model_path = "/data/asudupe/checkpoints/Latxa-3.1-8B-Omni/stage1/checkpoint-6992"
model_path = "/scratch/asudupe/checkpoints/Latxa-Llama-3.1-8B-Instruct/stage2/3950583/checkpoint-20973"
# model_path = "/hitz_data/asudupe/models/Latxa-Llama-3.1-8B-Instruct"
# model_path = "Llama-3.1-8B-Omni"
model_base = None
is_lora = False
s2s = True
mel_size = 128
conv_mode = 'llama_3'
tokenizer, model, context_len = load_pretrained_model(model_path, model_base, is_lora=is_lora, s2s=s2s)

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Loading checkpoint shards:  25%|██████████████████████▊                                                                    | 1/4 [00:01<00:03,  1.14s/it]

Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.15it/s]


In [5]:
hifigan = UnitHIFIGAN.from_hparams(source="/scratch/asudupe/models/hifigan/sonora_2/", run_opts={"device":'cuda'})

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [ ]:
qs = "<speech>\nPlease directly answer the questions in the user's speech."
speech_file = 'audioak/Recording 17.mp3'
# speech_file = os.path.join('/scratch/asudupe/datasets/VoiceAssistant-400K_eu',dataset['test'][1016]['question_audio'])
speech_loaded = whisper.load_audio(speech_file)
# audio = dataset['train'][20]['question_audio']
# speech = torch.tensor(audio, dtype=torch.float32)
# speech = Resample(orig_freq=22050, new_freq=16000)(speech)

conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()

speech = whisper.pad_or_trim(speech_loaded)
speech = whisper.log_mel_spectrogram(speech, n_mels=mel_size).permute(1, 0)
# speech = hubert_tokenizer(speech_loaded, sampling_rate=16000, return_tensors="pt", padding=True)['input_values'].permute(1, 0)

input_ids = tokenizer_speech_token(prompt, tokenizer, return_tensors='pt')
speech_length = torch.LongTensor([speech.shape[0]])

input_ids = input_ids.to(device='cuda', non_blocking=True)
speech_tensor = speech.to(dtype=torch.float16, device='cuda', non_blocking=True)
speech_length = speech_length.to(device='cuda', non_blocking=True)

input_ids = input_ids.unsqueeze(0)
speech_tensors = speech_tensor.unsqueeze(0)
speech_lengths = speech_length.unsqueeze(0)

# input_ids = torch.stack((input_ids), dim=0)
# speech_tensors = torch.stack((speech_tensor), dim=0)
# speech_lengths = torch.stack((speech_length), dim=0)

#torch.Size([1, 62]),torch.Size([1, 3000, 128]) #tensor([[3000]])
Audio(speech_loaded, rate=16000)

In [7]:
input_ids.shape, speech_tensors.shape, speech_lengths

(torch.Size([1, 64]),
 torch.Size([1, 3000, 128]),
 tensor([[3000]], device='cuda:0'))

In [ ]:
temperature = 0
top_p = None
num_beams = 1
max_new_tokens = 512

with torch.inference_mode():
    outputs = model.generate(
        input_ids,
        speech=speech_tensors,
        speech_lengths=speech_lengths,
        do_sample=True if temperature > 0 else False,
        temperature=temperature,
        top_p=top_p,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        pad_token_id=128004,
        streaming_unit_gen=True,
 
    )
# output_ids = outputs
output_ids, output_units = outputs

print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip())
output_units = ctc_postprocess(output_units, blank=model.config.unit_vocab_size)
output_units = torch.tensor([int(x) for x in output_units.split()], dtype=torch.long)
answer = hifigan.decode_unit(output_units.unsqueeze(-1), torch.tensor(np.load('/scratch/asudupe/models/hifigan/sonora_2/alex.npy')))
Audio(answer.cpu(), rate=16000)

Noski! Hona hemen zure txekiar errepublikar eguneko plana:  1. **Goiza**: Has zaitez Pragako Gaztelua bisitatuz, bere arkitektura ederra eta garrantzi historikoa miretsiz. 2. **Beranduago goizean**: Joan Karlos IV.aren Zubira, ikuspegi ikusgarriak eta garrantzi historikoa dituen monumentu historikoa. 3. **Arratsaldea**: Gozatu paseo lasai bat Hiri Zaharreko Plaza Historikoan, non Erloju Astronomikoa eta Týn Eliza ikus ditzakezun. 4. **Beranduago arratsaldean**: Amaitu eguna John Lennon Horma bisitatuz, non John Lennonen mural ikonikoa mirets dezakezun.  Ondo pasa bidaian!

In [ ]:
torchaudio.save("audioak/erantzuna.wav", answer.cpu(), sample_rate=16000)

In [7]:
import glob

In [8]:
speech_files = glob.glob('ebaluazioa/*')
for speech_file in speech_files:

    qs = "<speech>\nPlease directly answer the questions in the user's speech."
    speech_loaded = whisper.load_audio(speech_file)
    # audio = dataset['train'][20]['question_audio']
    # speech = torch.tensor(audio, dtype=torch.float32)
    # speech = Resample(orig_freq=22050, new_freq=16000)(speech)

    conv = conv_templates[conv_mode].copy()
    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    speech = whisper.pad_or_trim(speech_loaded)
    speech = whisper.log_mel_spectrogram(speech, n_mels=mel_size).permute(1, 0)
    # speech = hubert_tokenizer(speech_loaded, sampling_rate=16000, return_tensors="pt", padding=True)['input_values'].permute(1, 0)

    input_ids = tokenizer_speech_token(prompt, tokenizer, return_tensors='pt')
    speech_length = torch.LongTensor([speech.shape[0]])

    input_ids = input_ids.to(device='cuda', non_blocking=True)
    speech_tensor = speech.to(dtype=torch.float16, device='cuda', non_blocking=True)
    speech_length = speech_length.to(device='cuda', non_blocking=True)

    input_ids = input_ids.unsqueeze(0)
    speech_tensors = speech_tensor.unsqueeze(0)
    speech_lengths = speech_length.unsqueeze(0)

    # input_ids = torch.stack((input_ids), dim=0)
    # speech_tensors = torch.stack((speech_tensor), dim=0)
    # speech_lengths = torch.stack((speech_length), dim=0)

    #torch.Size([1, 62]),torch.Size([1, 3000, 128]) #tensor([[3000]])

    temperature = 0
    top_p = None
    num_beams = 1
    max_new_tokens = 512

    with torch.inference_mode():
        outputs = model.generate(
            input_ids,
            speech=speech_tensors,
            speech_lengths=speech_lengths,
            do_sample=True if temperature > 0 else False,
            temperature=temperature,
            top_p=top_p,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            pad_token_id=128004,
            streaming_unit_gen=True,
    
        )
    # output_ids = outputs
    output_ids, output_units = outputs

    print(tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip())
    output_units = ctc_postprocess(output_units, blank=model.config.unit_vocab_size)
    output_units = torch.tensor([int(x) for x in output_units.split()], dtype=torch.long)
    answer = hifigan.decode_unit(output_units.unsqueeze(-1), torch.tensor(np.load('/scratch/asudupe/models/hifigan/sonora_2/alex.npy')))

    speech_file = speech_file.split('/')[-1].split('.')[0]
    torchaudio.save("erantzuna_latxa_omni/"+speech_file+".wav", answer.cpu(), sample_rate=16000)

/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
The attention layers in this model are transitioning from computing the RoPE embeddings internally through `position_ids` (2D tensor with the indexes of the tokens), to using externally computed `position_embeddings` (Tuple of tensors, containing cos and sin). In v4.46 `position_ids` will be removed and `position_embeddings` will be mandatory.


Zure etxerako Gabonetako apaingarriak aukeratzea benetan dibertigarria izan daiteke! Kontuan hartu beharreko ezinbesteko gauza batzuk hauek dira: 1. Zure sarrerako atean zintzilikatzeko girlandak edo koroak. 2. Zure zuhaitza edo zuhaitz artifiziala apaintzeko argiak. 3. Zure etxeko tximinian zintzilikatzeko galtzerdiak. 4. Zure eskaileretan edo eskudelak apaintzeko girlanda. 5. Zure mahaietan jartzeko erdiguneak edo kandelak. 6. Zure etxeko txokoetan jartzeko apaingarri txikiak edo dekorazio-pieza bereziak. 7. Zure lorategian edo atarian jartzeko kanpoko apaingarriak. Elementu hauek ukitu festibo bat emango diote zure etxeari eta oporraldi alaia sortuko dute!


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Jakina! Zuretzat egokia den hidriderik onena zure behar zehatzen araberakoa da. Etxeko erabilera orokorrerako, ura eta gatzaren nahasketa bat edo ura eta elektrolitoen soluzio bat aukera ona izan daiteke. Jarduera fisikoetarako edo elektrolito gehiago behar badituzu, kirol-hidrider bat kontuan hartu dezakezu. Larruazal sentikorra baduzu, bilatu lurrinik gabeko aukerak. Garrantzitsua da zaporea eta osagaiak egiaztatzea, zure lehentasunak eta osasun-baldintzak betetzen dituztela ziurtatzeko.


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


...

Adimen artifizialarekin lotutako dilema etiko baten adibidea da erabakiak hartzeko prozesuetan AI sistemek duten gardentasuna. AI sistemek erabakiak nola hartzen dituzten ulertzea erronka izan daiteke, askotan algoritmo konplexuetan eta datu kopuru handietan oinarritzen baitira. Horrek zaildu egiten du erantzukizuna zehaztea eta sistemak modu bidezkoan eta alborapenik gabe funtzionatzen dutela bermatzea. Gainera, kezka dago lanpostuen ordezkapenari buruz, AIk industria jakin batzuetan langileak ordezka baititzake, langabezia eta desberdintasun ekonomikoa eraginez.


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Benetan sentitzen dut horrela sentitzen zarela, baina ezin dizut horretan lagundu. Garrantzitsua da laguntza eman diezazukeen norbaitekin hitz egitea, hala nola konfiantzazko lagun, senide edo osasun mentaleko profesional batekin.


/scratch/asudupe/conda-env/latxa-omni/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Bai, badaude animalia berezi edo exotikoak maskota gisa eduki ditzakezunak, baina garrantzitsua da lehenik tokiko legeak eta araudiak egiaztatzea. Animalia exotiko batzuk, hala nola azukre-hegaztiak, trikuak eta fennec azeriak, maskota gisa edukitzen dira leku batzuetan. Hala ere, animalia horiek zainketa-betekizun zehatzak izan ditzakete eta arreta berezia behar dute. Gainera, funtsezkoa da ziurtatzea animalia horiek modu etikoan eta gizalegez tratatzen direla. Tokiko basa-bizitzako agintariekin edo maskota exotikoen adituekin kontsultatzeak informazio zehatzagoa eman dezake zure eremuan maskota exotikoak edukitzearen inguruan.
